# Notebook 1 — Conceitos e Componentes de Séries Temporais

**Disciplina:** Estatística II — Ciência de Dados
**Fatec — 2026**

---

## Objetivos de aprendizagem

Ao final deste notebook, você será capaz de:

1. Identificar uma série temporal e seus **componentes** (tendência, ciclo, sazonalidade e ruído).
2. Distinguir **modelos aditivo e multiplicativo**.
3. Calcular **média móvel** manualmente e com o Pandas.
4. Interpretar o efeito do tamanho da **janela** de suavização.

## Como usar este notebook

Este notebook é **interativo**: em vez de só clicar em "Executar", você vai:

- Prever resultados **antes** de rodar o código;
- Alterar parâmetros e observar o efeito;
- Completar trechos de código (`# SEU CÓDIGO AQUI`);
- Responder perguntas de interpretação nas células de texto.

> **Não pule as células de predição!** O aprendizado acontece quando você tenta adivinhar o resultado antes de ver a resposta.

---

## 1. O que é uma série temporal?

Uma **série temporal** é um conjunto sequencial de observações medidas em **tempos sucessivos e regulares** (ano, mês, dia, hora…).

Três componentes definem uma série temporal:

| Componente | Exemplo |
|---|---|
| **Medida** (o que) | Total de pessoas |
| **Fato** (em relação a quê) | atendidas em um hospital |
| **Unidade de tempo** (quando) | por dia |

> **Por que a ordem importa?** Em uma série temporal, os valores **dependem do tempo**. Embaralhar as observações destrói a informação.

### Setup do ambiente

In [ ]:
# Bibliotecas que vamos usar neste notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Deixa os gráficos um pouco maiores e com grade
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Fixa a semente aleatória para todos verem o mesmo resultado
np.random.seed(42)

print("Tudo pronto!")

---

## 2. Exercício de predição: reconhecendo componentes

Vamos começar com um exercício **antes de ver qualquer código executado**.

Você vai rodar quatro simulações: cada uma tem um ou mais dos seguintes componentes — **tendência**, **sazonalidade**, **ciclo**, **ruído**.

### ✏️ Sua tarefa ANTES de executar a próxima célula

Na célula de texto abaixo, escreva **o que você espera ver em cada um dos 4 gráficos**. Não se preocupe em acertar — o importante é prever.

---

### Minha predição (preencha antes de rodar a próxima célula)

- **Gráfico 1** (`serie_A`): _______________________
- **Gráfico 2** (`serie_B`): _______________________
- **Gráfico 3** (`serie_C`): _______________________
- **Gráfico 4** (`serie_D`): _______________________

Dica: para cada um, pense se vai ter **tendência** (subida/descida de longo prazo), **sazonalidade** (repetição regular), **ciclo** (ondas largas) e/ou **ruído** (variação aleatória).

In [ ]:
# Vamos gerar 4 séries temporais diferentes. Cada uma combina componentes diferentes.
# Execute e compare com sua predição.

t = np.arange(1, 121)  # 120 meses = 10 anos

# Série A: só tendência
serie_A = 0.5 * t

# Série B: tendência + sazonalidade
serie_B = 0.5 * t + 10 * np.sin(2 * np.pi * t / 12)

# Série C: tendência + sazonalidade + ruído
serie_C = 0.5 * t + 10 * np.sin(2 * np.pi * t / 12) + np.random.normal(0, 3, 120)

# Série D: só ruído
serie_D = np.random.normal(0, 3, 120)

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
axes[0,0].plot(t, serie_A, color='steelblue'); axes[0,0].set_title('Série A')
axes[0,1].plot(t, serie_B, color='darkgreen'); axes[0,1].set_title('Série B')
axes[1,0].plot(t, serie_C, color='darkorange'); axes[1,0].set_title('Série C')
axes[1,1].plot(t, serie_D, color='crimson'); axes[1,1].set_title('Série D')
for ax in axes.flat:
    ax.set_xlabel('Mês'); ax.set_ylabel('Valor')
plt.tight_layout()
plt.show()

### 🔍 Comparando com a resposta

A série A tem apenas **tendência** linear. A série B adiciona **sazonalidade** (um ciclo regular de 12 meses). A série C adiciona **ruído** aleatório — é a que mais se parece com dados reais. A série D tem **apenas ruído** (sem tendência nem padrão).

**Sua predição bateu?** Escreva abaixo o que você errou e por quê.

### Minha reflexão

_(escreva 2-3 linhas sobre o que você aprendeu com este exercício)_



---

## 3. Os componentes, um a um

Vamos agora **construir** uma série temporal somando componentes — como montar uma receita.

### 3.1 Tendência (T)

Movimento de longo prazo: crescimento, queda ou estabilidade. Exemplo: crescimento populacional.

In [ ]:
t = np.arange(1, 61)  # 60 meses = 5 anos

tendencia = 0.8 * t   # cresce 0.8 unidades por mês

plt.plot(t, tendencia, color='black', linewidth=2)
plt.title('Componente: Tendência (linear crescente)')
plt.xlabel('Mês'); plt.ylabel('Valor')
plt.show()

### 3.2 Sazonalidade (S)

Padrão **regular e previsível** que se repete em intervalos fixos. Exemplo: vendas de sorvete sobem todo dezembro.

Note que a sazonalidade oscila **em torno de zero** — ela é um "ajuste" que se soma ou subtrai da tendência.

In [ ]:
sazonalidade = 10 * np.sin(2 * np.pi * t / 12)  # ciclo de 12 meses

plt.plot(t, sazonalidade, color='darkorange', linewidth=2)
plt.title('Componente: Sazonalidade (ciclo anual)')
plt.xlabel('Mês'); plt.ylabel('Valor')
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.show()

### ✏️ Experimento

Altere o código acima trocando o `12` por **outros valores** (6, 4, 3). O que representa esse número no mundo real?

Escreva sua resposta aqui:

_________________________________________________

### 3.3 Ruído (R)

Variações **aleatórias** sem padrão. É o que sobra depois de explicar tendência, sazonalidade e ciclo.

In [ ]:
ruido = np.random.normal(loc=0, scale=2, size=60)

plt.plot(t, ruido, color='gray', linewidth=1)
plt.title('Componente: Ruído (aleatório)')
plt.xlabel('Mês'); plt.ylabel('Valor')
plt.axhline(0, color='red', linestyle='--', alpha=0.5)
plt.show()

### 3.4 Juntando tudo — modelo aditivo

$$Y(t) = T(t) + S(t) + R(t)$$

Esse modelo funciona quando as **flutuações sazonais têm amplitude constante**.

In [ ]:
serie_aditiva = tendencia + sazonalidade + ruido

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
axes[0].plot(t, tendencia, color='black'); axes[0].set_ylabel('Tendência')
axes[1].plot(t, sazonalidade, color='darkorange'); axes[1].set_ylabel('Sazonalidade')
axes[2].plot(t, ruido, color='gray'); axes[2].set_ylabel('Ruído')
axes[3].plot(t, serie_aditiva, color='steelblue', linewidth=2); axes[3].set_ylabel('Série final')
axes[3].set_xlabel('Mês')
fig.suptitle('Modelo Aditivo: Y(t) = T(t) + S(t) + R(t)', fontsize=14)
plt.tight_layout()
plt.show()

---

## 4. Aditivo vs Multiplicativo — experimente os parâmetros

No modelo **multiplicativo**, os componentes se multiplicam:

$$Y(t) = T(t) \times S(t) \times R(t)$$

Use-o quando a **amplitude das oscilações cresce junto com o nível da série**.

### 🎛️ Atividade interativa

Use o formulário abaixo para gerar séries aditivas e multiplicativas com diferentes parâmetros. **Rode várias vezes** variando os valores.

In [ ]:
#@title Experimento: Aditivo vs Multiplicativo { run: "auto" }
amplitude_sazonal = 7 #@param {type:"slider", min:1, max:30, step:1}
inclinacao_tendencia = 1.1 #@param {type:"slider", min:0.1, max:2.0, step:0.1}
nivel_ruido = 2 #@param {type:"slider", min:0, max:10, step:1}

t = np.arange(1, 61)
np.random.seed(42)

# Componentes
T = inclinacao_tendencia * t + 5              # tendência (sempre positiva)
S_add = amplitude_sazonal * np.sin(2*np.pi*t/12)       # sazonalidade aditiva
S_mult = 1 + (amplitude_sazonal/50) * np.sin(2*np.pi*t/12)  # sazonalidade multiplicativa (~1)
R_add = np.random.normal(0, nivel_ruido, 60)
R_mult = np.random.normal(1, nivel_ruido/20, 60)

Y_aditiva = T + S_add + R_add
Y_multiplicativa = T * S_mult * R_mult

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(t, Y_aditiva, color='steelblue')
ax1.set_title('Modelo Aditivo')
ax1.set_xlabel('Mês'); ax1.set_ylabel('Valor')
ax2.plot(t, Y_multiplicativa, color='darkgreen')
ax2.set_title('Modelo Multiplicativo')
ax2.set_xlabel('Mês'); ax2.set_ylabel('Valor')
plt.tight_layout()
plt.show()

### 🧠 Perguntas para reflexão

Rode a célula acima variando os sliders e responda:

1. **Aumente `amplitude_sazonal` de 1 para 30.** O que muda no gráfico da esquerda? E no da direita?

   _Sua resposta:_ ______________________________________________

2. **Aumente `inclinacao_tendencia`.** Em qual dos dois modelos a amplitude das oscilações também cresce?

   _Sua resposta:_ ______________________________________________

3. Qual modelo você usaria para modelar:
   - **Vendas de uma pizzaria** que crescem ao longo dos anos, com pico mensal no fim de semana? → ___________
   - **Temperatura média** de São Paulo (flutuações anuais constantes)? → ___________

### 💡 Caso real: o IPCA brasileiro

O **IPCA** (Índice Nacional de Preços ao Consumidor Amplo) mede a inflação no Brasil. Vamos carregar a série mensal direto da API do Banco Central e analisar seu comportamento.

> **Dica Colab:** a célula abaixo carrega dados da web. Se estiver lento, aguarde 5-10 segundos.

In [ ]:
# Carregando IPCA mensal do Banco Central (variação % mensal)
url_ipca = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01/01/2010&dataFinal=31/12/2024"

ipca = pd.read_json(url_ipca)
ipca['data'] = pd.to_datetime(ipca['data'], format='%d/%m/%Y')
ipca = ipca.set_index('data')
ipca.columns = ['IPCA (% mensal)']

print(f"Dados: {len(ipca)} observações mensais, de {ipca.index.min().date()} a {ipca.index.max().date()}")
ipca.head()

In [ ]:
# Visualize a série
plt.figure(figsize=(12, 4))
plt.plot(ipca.index, ipca['IPCA (% mensal)'], color='darkblue')
plt.title('IPCA Mensal — Brasil (Fonte: BCB)')
plt.ylabel('Variação % mensal')
plt.xlabel('Ano')
plt.axhline(0, color='gray', linestyle='--', alpha=0.5)
plt.show()

### ✏️ Interprete

Olhando o gráfico do IPCA:

1. Você observa uma **tendência** clara de longo prazo? ___________________
2. Existe **sazonalidade**? Se sim, quantos meses parece ter o ciclo? _______________
3. Qual modelo parece mais adequado, **aditivo** ou **multiplicativo**? Por quê? ________________________

---

## 5. Média Móvel — implementando do zero

A **média móvel** é uma técnica de **suavização**: reduz o ruído e deixa a tendência mais visível.

Em vez de pegar a função pronta do Pandas, vamos **construí-la à mão** primeiro — é a melhor forma de entender o que está acontecendo.

### 5.1 Conceito

Para cada ponto da série, a média móvel é a **média dos últimos N valores** (incluindo o ponto atual). N é o **tamanho da janela**.

Exemplo com janela de 3:

| Dia | Vendas | Média Móvel (3) |
|-----|--------|-----------------|
| Seg | 30     | — (faltam dados)|
| Ter | 28     | — (faltam dados)|
| Qua | 35     | (30+28+35)/3 = 31.0 |
| Qui | 33     | (28+35+33)/3 = 32.0 |
| Sex | 50     | (35+33+50)/3 = 39.3 |

### ✏️ Complete a função abaixo

In [ ]:
def minha_media_movel(serie, janela):
    """Calcula a média móvel de uma série numérica.

    Parâmetros:
    - serie: lista ou array de números
    - janela: inteiro, tamanho da janela

    Retorna:
    - uma lista do mesmo tamanho da série, com NaN nas primeiras (janela-1) posições
    """
    resultado = []
    for i in range(len(serie)):
        if i < janela - 1:
            # Não há dados suficientes para calcular a média
            resultado.append(np.nan)
        else:
            # SEU CÓDIGO AQUI:
            # pegue os últimos 'janela' valores terminando em i (inclusive)
            # calcule a média e adicione a 'resultado'
            janela_atual = ________________  # dica: use serie[i - janela + 1 : i + 1]
            media = ________________         # dica: np.mean(...)
            resultado.append(media)
    return resultado

# Teste com os dados do exemplo
vendas = [30, 28, 35, 33, 50, 60, 55]
mm3 = minha_media_movel(vendas, janela=3)
print("Valores obtidos:", [f"{x:.1f}" if not np.isnan(x) else "NaN" for x in mm3])
print("Valores esperados: ['NaN', 'NaN', '31.0', '32.0', '39.3', '47.7', '55.0']")

### 🧪 Teste sua implementação

Se sua função estiver correta, a célula abaixo não levantará erro:

In [ ]:
# Teste automático
vendas = [30, 28, 35, 33, 50, 60, 55]
mm3 = minha_media_movel(vendas, janela=3)

assert np.isnan(mm3[0]), "Os primeiros valores devem ser NaN"
assert np.isnan(mm3[1]), "Os primeiros valores devem ser NaN"
assert abs(mm3[2] - 31.0) < 0.1, f"mm3[2] deveria ser 31.0, mas veio {mm3[2]}"
assert abs(mm3[5] - 47.67) < 0.1, f"mm3[5] deveria ser ~47.67, mas veio {mm3[5]}"
print("✅ Implementação correta!")

---

## 6. Comparando com o Pandas `.rolling()`

Agora que você entendeu o que acontece por dentro, o Pandas faz o mesmo em uma linha:

In [ ]:
# Comparação: sua função vs Pandas
vendas_series = pd.Series(vendas)
mm3_pandas = vendas_series.rolling(window=3).mean()

comparacao = pd.DataFrame({
    'Vendas': vendas,
    'Minha função': minha_media_movel(vendas, 3),
    'Pandas .rolling()': mm3_pandas.values
})
comparacao

### 6.1 Aplicando no IPCA

Vamos suavizar a série do IPCA para enxergar melhor a tendência.

In [ ]:
# Média móvel de 12 meses (um ano completo) — filtra sazonalidade mensal
ipca['MM12'] = ipca['IPCA (% mensal)'].rolling(window=12).mean()

plt.figure(figsize=(12, 4))
plt.plot(ipca.index, ipca['IPCA (% mensal)'], color='lightgray', label='IPCA mensal')
plt.plot(ipca.index, ipca['MM12'], color='red', linewidth=2, label='Média Móvel 12 meses')
plt.title('IPCA com suavização de 12 meses')
plt.xlabel('Ano'); plt.ylabel('% mensal')
plt.legend()
plt.show()

### 🎛️ Experimento: o efeito da janela

Teste diferentes tamanhos de janela e veja como a suavização muda.

In [ ]:
#@title Tamanho da janela { run: "auto" }
janela = 19 #@param {type:"slider", min:1, max:36, step:1}

mm = ipca['IPCA (% mensal)'].rolling(window=janela).mean()

plt.figure(figsize=(12, 4))
plt.plot(ipca.index, ipca['IPCA (% mensal)'], color='lightgray', label='Original')
plt.plot(ipca.index, mm, color='red', linewidth=2, label=f'MM {janela} meses')
plt.title(f'IPCA suavizado — janela de {janela} meses')
plt.legend()
plt.ylabel('% mensal')
plt.show()

### 🧠 Perguntas de interpretação

1. **Janela = 1.** O que acontece? Por quê? ___________________________

2. **Janela = 3** vs **Janela = 24.** Qual janela mostra melhor a **tendência** de longo prazo? Qual preserva mais o **ruído**? ___________________________

3. **Trade-off:** o que você ganha e o que você perde ao aumentar a janela? ___________________________

4. **Aplicação prática:** se seu objetivo é *detectar rapidamente* uma mudança de tendência, você prefere janela pequena ou grande? ___________________________

---

## 7. Desafio final — crie sua própria série temporal

Agora é com você. Crie uma série temporal **simulando vendas de uma sorveteria em São Paulo** ao longo de 3 anos (36 meses) com as seguintes características:

- Tendência de crescimento de **2 unidades por mês** (a sorveteria está ganhando clientela)
- Sazonalidade anual: pico em **janeiro** (verão) e vale em **julho** (inverno), amplitude de **15 unidades**
- Ruído aleatório com desvio padrão de **3 unidades**
- Nível inicial de **50 unidades**

Depois, aplique uma média móvel de 3 meses e plote tudo junto.

> **Dica:** use `np.sin(2*np.pi*(t-1)/12)` para o pico em janeiro (t=1).

### ✏️ Seu código

In [ ]:
# SEU CÓDIGO AQUI

# 1. Crie o vetor de tempo (36 meses)


# 2. Construa os componentes


# 3. Some tudo (modelo aditivo)


# 4. Calcule a média móvel de 3 meses


# 5. Plote a série original + média móvel



### ✏️ Relatório

Depois de rodar seu código, responda:

1. A média móvel conseguiu **remover** completamente a sazonalidade? Por quê?

   _______________________________________________________________

2. Que tamanho de janela **removeria** toda a sazonalidade anual, mantendo só a tendência? (dica: pense no ciclo)

   _______________________________________________________________

3. Se você fosse o dono da sorveteria, que **previsão** faria para os próximos 12 meses com base nessa série?

   _______________________________________________________________